# [Chapter 2 — The Infection Equation, §2.1-2.3] Compartments, Contact Rates, and the Infection Equation

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 2 — The Infection Equation
**Considerations developed:** 2 (compartmental simplifications), 3 (equal-contact-rate)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Verifies the core infection-equation identity $\lambda S = \alpha I$ algebraically and numerically, and shows what the equal-contact-rate simplification ($c_S = c_I = c_R$) erases from the model.

## What you should already know
Basic ODE familiarity; the SIR compartment idea.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import set_seed_chapter_02, book_style, BOOK_COLORS
from shared.verification import assert_within_tolerance

set_seed_chapter_02()
book_style()


## The infection-equation identity

The book defines two views of the same incidence rate:

- **Force of infection** (susceptible viewpoint): $\lambda = c_S \beta P_I$ where $P_I = I/N$
- **Force from infection** (infected viewpoint): $\alpha = c_I \beta P_S$ where $P_S = S/N$

Incidence is $J = \lambda S = \alpha I$. Algebraically:

$$\lambda S = c_S \beta \frac{I}{N} S = c_S \beta \frac{IS}{N}$$
$$\alpha I = c_I \beta \frac{S}{N} I = c_I \beta \frac{IS}{N}$$

These are equal **only when** $c_S = c_I$. Let's check both cases.

In [ ]:
# Case 1: c_S = c_I (the equal-contact-rate simplification)
N = 1000
S = 800
I = 50
c_S = c_I = 10.0  # equal
beta = 0.025

lambda_ = c_S * beta * (I / N)
alpha = c_I * beta * (S / N)

incidence_from_lambda = lambda_ * S
incidence_from_alpha = alpha * I

print(f"Case 1: c_S = c_I = {c_S}")
print(f"  lambda * S = {incidence_from_lambda:.4f}")
print(f"  alpha * I  = {incidence_from_alpha:.4f}")
assert_within_tolerance(incidence_from_lambda, incidence_from_alpha, rel_tol=1e-12)
print("  ✅ identity verified")


In [ ]:
# Case 2: c_S ≠ c_I (the realistic case the book uses)
c_S = 10.0
c_I = 8.0   # infectious individuals reduce contact (e.g., self-isolation)

lambda_ = c_S * beta * (I / N)
alpha = c_I * beta * (S / N)

incidence_from_lambda = lambda_ * S
incidence_from_alpha = alpha * I

print(f"Case 2: c_S = {c_S}, c_I = {c_I}")
print(f"  lambda * S = {incidence_from_lambda:.4f}")
print(f"  alpha * I  = {incidence_from_alpha:.4f}")

# These are NOT equal — and that's the point.
diff = abs(incidence_from_lambda - incidence_from_alpha)
print(f"  difference: {diff:.4f}")
print(f"  ratio: {incidence_from_lambda / incidence_from_alpha:.3f}")


## Why the unequal-contact-rate case matters

The difference between $c_S$ and $c_I$ is exactly where most realistic interventions act:

- **Case isolation, quarantine, hospitalization** all reduce $c_I$ specifically
- **Behavioral changes following diagnosis** reduce $c_I$ specifically
- **Symptom-triggered withdrawal** reduces $c_I$ specifically

A model that assumes $c_S = c_I$ cannot represent the effect of any of these interventions. This is **Consideration 3** — the equal-contact-rate simplification erases the parameter with the largest invasion sensitivity.

The book's convention throughout is to allow $c_S \neq c_I \neq c_R$, and to use the **infected-viewpoint** form $\alpha = c_I \beta P_S$ as the primary expression.

In [ ]:
# Visualize the c_S/c_I asymmetry effect on R_0
c_I_range = np.linspace(2, 12, 50)
beta = 0.025
tau_R = 10
R0 = c_I_range * beta * tau_R

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(c_I_range, R0, color=BOOK_COLORS['primary'], lw=2)
ax.axhline(1.0, color=BOOK_COLORS['negative'], ls='--', lw=1, alpha=0.7, label='Invasion threshold $R_0 = 1$')
ax.fill_between(c_I_range, 0, R0, where=(R0 < 1), alpha=0.1, color=BOOK_COLORS['negative'])
ax.fill_between(c_I_range, 0, R0, where=(R0 >= 1), alpha=0.1, color=BOOK_COLORS['positive'])
ax.set_xlabel('$c_I$ (infectious-class contact rate)')
ax.set_ylabel('$R_0 = c_I \\beta \\tau_R$')
ax.set_title('$R_0$ depends only on $c_I$, not $c_S$ or $c_R$')
ax.legend()
ax.set_xlim(2, 12)
plt.tight_layout()
plt.show()

print(f"At c_I = 4:  R_0 = {4 * beta * tau_R:.2f}  (would invade)")
print(f"At c_I = 8:  R_0 = {8 * beta * tau_R:.2f}  (will invade)")
print(f"At c_I = 12: R_0 = {12 * beta * tau_R:.2f}  (will invade strongly)")


## What's next

The next two notebooks in this chapter develop the two viewpoints separately:

- **Chapter 2A** — Force of Infection (susceptible viewpoint, the conventional form)
- **Chapter 2B** — Force from Infection (infected viewpoint, the book's primary form)

The choice between these two viewpoints determines what is invariant under the data — the book's central methodological theme.